### Final Data Merge and Schema Finalization

This notebook completes the data preparation by adding final calculated columns and enforcing a final, standardized column order.

**Workflow:**

1.  **Load Data:** Loads the main DataFrame containing both Finviz data and previously calculated performance ratios.
2.  **Feature Engineering:** Calculates new features directly from the loaded data (e.g., `ATR/Price %`).
3.  **Merge External Data:** Calculates the 3-day performance from the separate OHLCV data file and merges this new column into the main DataFrame.
4.  **Finalize Schema:** Reorders all columns according to a predefined master list to ensure a consistent output format.
5.  **Save & Verify:** Saves the final, completed DataFrame and reads it back to confirm success.

### Setup and Configuration

This cell loads all necessary libraries and configuration parameters. It pulls dynamic settings from `config.py` and defines the final column schema.


In [1]:
import sys
from pathlib import Path
import pandas as pd

# --- Project Path Setup ---
NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# --- Dynamic Configuration (from config.py) ---
from config import DATE_STR, DEST_DIR

# --- File Path Construction ---
DATA_DIR = Path(DEST_DIR)
SOURCE_PATH = DATA_DIR / f'{DATE_STR}_df_finviz_n_ratios_stocks_etfs.parquet'
OHLCV_PATH = DATA_DIR / 'df_OHLCV_clean_stocks_etfs.parquet'
DEST_PATH = DATA_DIR / f'{DATE_STR}_df_finviz_merged_stocks_etfs.parquet'

# --- Final Schema Configuration ---
# This list defines the exact order of columns in the final output file.
FINAL_COLUMN_ORDER = [
    'No.', 'Company', 'Index', 'Sector', 'Industry', 'Country', 'Exchange',
    'Info', 'MktCap AUM, M', 'Rank', 'Market Cap, M', 'P/E', 'Fwd P/E', 'PEG',
    'P/S', 'P/B', 'P/C', 'P/FCF', 'Book/sh', 'Cash/sh', 'Dividend %',
    'Dividend TTM', 'Dividend Ex Date', 'Payout Ratio %', 'EPS', 'EPS next Q',
    'EPS this Y %', 'EPS next Y %', 'EPS past 5Y %', 'EPS next 5Y %',
    'Sales past 5Y %', 'Sales Q/Q %', 'EPS Q/Q %', 'EPS YoY TTM %',
    'Sales YoY TTM %', 'Sales, M', 'Income, M', 'EPS Surprise %',
    'Revenue Surprise %', 'Outstanding, M', 'Float, M', 'Float %',
    'Insider Own %', 'Insider Trans %', 'Inst Own %', 'Inst Trans %',
    'Short Float %', 'Short Ratio', 'Short Interest, M', 'ROA %', 'ROE %',
    # 'ROI %', 'Curr R', 'Quick R', 'LTDebt/Eq', 'Debt/Eq', 'Gross M %',
    'ROIC %', 'Curr R', 'Quick R', 'LTDebt/Eq', 'Debt/Eq', 'Gross M %',    
    'Oper M %', 'Profit M %', 'Perf 3D %', 'Perf Week %', 'Perf Month %',
    'Perf Quart %', 'Perf Half %', 'Perf Year %', 'Perf YTD %', 'Beta',
    'ATR', 'ATR/Price %', 'Volatility W %', 'Volatility M %', 'SMA20 %',
    'SMA50 %', 'SMA200 %', '50D High %', '50D Low %', '52W High %',
    '52W Low %', '52W Range', 'All-Time High %', 'All-Time Low %', 'RSI',
    'Earnings', 'IPO Date', 'Optionable', 'Shortable', 'Employees',
    'Change from Open %', 'Gap %', 'Recom', 'Avg Volume, M', 'Rel Volume',
    'Volume', 'Target Price', 'Prev Close', 'Open', 'High', 'Low', 'Price',
    'Change %', 'Single Category', 'Asset Type', 'Expense %', 'Holdings',
    'AUM, M', 'Flows 1M, M', 'Flows% 1M', 'Flows 3M, M', 'Flows% 3M',
    'Flows YTD, M', 'Flows% YTD', 'Return% 1Y', 'Return% 3Y', 'Return% 5Y',
    'Tags', 'Sharpe 3d', 'Sortino 3d', 'Omega 3d', 'Sharpe 5d',
    'Sortino 5d', 'Omega 5d', 'Sharpe 10d', 'Sortino 10d', 'Omega 10d',
    'Sharpe 15d', 'Sortino 15d', 'Omega 15d', 'Sharpe 30d',
    'Sortino 30d', 'Omega 30d', 'Sharpe 60d', 'Sortino 60d', 'Omega 60d',
    'Sharpe 120d', 'Sortino 120d', 'Omega 120d', 'Sharpe 250d',
    'Sortino 250d', 'Omega 250d'
]

# --- Notebook Setup ---
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
%load_ext autoreload
%autoreload 2

# --- Verification ---
print(f"Processing for Date: {DATE_STR}")
print(f"Source file: {SOURCE_PATH}")
print(f"OHLCV source for 3D Perf: {OHLCV_PATH}")
print(f"Destination file: {DEST_PATH}")

Processing for Date: 2026-07-17
Source file: c:\Users\ping\Files_win10\python\py311\stocks\data\2026-07-17_df_finviz_n_ratios_stocks_etfs.parquet
OHLCV source for 3D Perf: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_clean_stocks_etfs.parquet
Destination file: c:\Users\ping\Files_win10\python\py311\stocks\data\2026-07-17_df_finviz_merged_stocks_etfs.parquet


### Step 1: Load Source Data

Load the main DataFrame containing the combined Finviz and performance ratio data.


In [2]:
print(f"--- Step 1: Loading data from {SOURCE_PATH.name} ---")

try:
    df_finviz = pd.read_parquet(SOURCE_PATH)
    # The list of tickers is derived directly from our primary source file.
    tickers = df_finviz.index.tolist()
    print(f"Successfully loaded data for {len(tickers)} tickers.")
    df_finviz.info()

except FileNotFoundError:
    print(f"ERROR: Source file not found at {SOURCE_PATH}. Halting execution.")
    df_finviz = None
except Exception as e:
    print(f"An error occurred during file loading: {e}")
    df_finviz = None

--- Step 1: Loading data from 2026-07-17_df_finviz_n_ratios_stocks_etfs.parquet ---
Successfully loaded data for 14 tickers.
<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, VV to PPL
Columns: 137 entries, No. to Omega 250d
dtypes: float64(118), int64(3), object(16)
memory usage: 15.1+ KB


### Step 2: Feature Engineering

Calculate new columns based on the existing data in the DataFrame.

In [3]:
if df_finviz is not None:
    print("\n--- Step 2: Engineering new features from existing data ---")

    # Calculate ATR as a percentage of Price
    df_finviz["ATR/Price %"] = (df_finviz["ATR"] / df_finviz["Price"]) * 100
    print("Created 'ATR/Price %' column.")

    display(df_finviz[["ATR", "Price", "ATR/Price %"]].head())


--- Step 2: Engineering new features from existing data ---
Created 'ATR/Price %' column.


,ATR,Price,ATR/Price %
VV,8.33,358.56,2.323182
PPG,3.37,149.98,2.246966
TTD,2.07,123.60,1.674757
TT,0.67,21.81,3.071985
SSO,1.80,95.30,1.888772


### Step 3: Calculate and Merge External Data (`Perf 3D %`)

This step calculates the 3-day performance using the external OHLCV file and merges it into our main DataFrame.

In [4]:
def calculate_3d_performance(ohlcv_path: Path, ticker_list: list) -> pd.DataFrame:
    """
    Loads OHLCV data, calculates 3-day performance for a list of tickers,
    and returns a DataFrame ready for merging.
    """
    try:
        df_ohlcv = pd.read_parquet(ohlcv_path)
    except FileNotFoundError:
        print(
            f"ERROR: OHLCV file not found at {ohlcv_path}. Cannot calculate 3D performance."
        )
        return pd.DataFrame()

    # Pivot to wide format with tickers as columns
    df_adj_close = df_ohlcv["Adj Close"].unstack(level="Ticker")

    # Filter for tickers present in our main DataFrame
    valid_tickers = [t for t in ticker_list if t in df_adj_close.columns]
    df_adj_close_filtered = df_adj_close[valid_tickers]

    # Calculate 3-day percentage change and get the latest value
    df_returns = df_adj_close_filtered.pct_change(periods=3) * 100
    latest_returns = df_returns.tail(1)

    # Transpose and rename for merging
    df_perf_3d = latest_returns.T
    df_perf_3d.columns = ["Perf 3D %"]

    return df_perf_3d


if df_finviz is not None:
    print("\n--- Step 3: Calculating and merging 3-day performance ---")
    df_perf_3d = calculate_3d_performance(OHLCV_PATH, tickers)

    if not df_perf_3d.empty:
        # Join the new column to the main DataFrame
        df_merged = df_finviz.join(df_perf_3d)
        print("Successfully calculated and merged 'Perf 3D %'.")
        display(df_merged[["Perf 3D %"]].head())
    else:
        print("Could not calculate 3D performance, continuing without it.")
        df_merged = df_finviz  # Assign original df if calculation failed
else:
    print("Skipping merge step because source data did not load.")
    df_merged = None


--- Step 3: Calculating and merging 3-day performance ---
Successfully calculated and merged 'Perf 3D %'.


,Perf 3D %
VV,-1.241770
PPG,2.354788
TTD,-1.847941
TT,-2.806328
SSO,-2.391783


### Step 4: Finalize Schema by Reordering Columns

Enforce the final column order as defined in the `FINAL_COLUMN_ORDER` list.

In [5]:
if df_merged is not None:
    print("\n--- Step 4: Finalizing DataFrame schema ---")

    # Check for any columns in the master list that are missing from our DataFrame
    missing_cols = [col for col in FINAL_COLUMN_ORDER if col not in df_merged.columns]
    if missing_cols:
        print(
            f"Warning: The following columns from the master list are missing and will be added as empty: {missing_cols}"
        )

    # Reindex the DataFrame to match the final desired column order
    df_final = df_merged.reindex(columns=FINAL_COLUMN_ORDER)

    print("Columns have been reordered to the final schema.")
    df_final.info()
else:
    print("Skipping schema finalization as merged data is not available.")
    df_final = None


--- Step 4: Finalizing DataFrame schema ---
Columns have been reordered to the final schema.
<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, VV to PPL
Columns: 139 entries, No. to Omega 250d
dtypes: float64(120), int64(3), object(16)
memory usage: 15.9+ KB


### Step 5: Save and Verify Final DataFrame

Save the completed DataFrame and read it back to confirm the entire pipeline was successful.

In [6]:
if df_final is not None:
    print("\n--- Step 5: Saving and verifying final data ---")
    try:
        df_final.to_parquet(DEST_PATH, engine="pyarrow", compression="zstd")
        print(f"Successfully saved final DataFrame to: {DEST_PATH}")

        # Verification step
        print("\nVerifying saved file...")
        verified_df = pd.read_parquet(DEST_PATH)
        print("Verification successful. First 5 rows of final saved file:")
        display(verified_df.head())

    except Exception as e:
        print(f"An error occurred during save or verification: {e}")
else:
    print("\nSkipping final save step as the final DataFrame was not created.")


--- Step 5: Saving and verifying final data ---
Successfully saved final DataFrame to: c:\Users\ping\Files_win10\python\py311\stocks\data\2026-07-17_df_finviz_merged_stocks_etfs.parquet

Verifying saved file...
Verification successful. First 5 rows of final saved file:


,No.,Company,Index,Sector,Industry,Country,Exchange,Info,"MktCap AUM, M",Rank,"Market Cap, M",P/E,Fwd P/E,PEG,P/S,P/B,P/C,P/FCF,Book/sh,Cash/sh,Dividend %,Dividend TTM,Dividend Ex Date,Payout Ratio %,EPS,EPS next Q,EPS this Y %,EPS next Y %,EPS past 5Y %,EPS next 5Y %,Sales past 5Y %,Sales Q/Q %,EPS Q/Q %,EPS YoY TTM %,Sales YoY TTM %,"Sales, M","Income, M",EPS Surprise %,Revenue Surprise %,"Outstanding, M","Float, M",Float %,Insider Own %,Insider Trans %,Inst Own %,Inst Trans %,Short Float %,Short Ratio,"Short Interest, M",ROA %,ROE %,ROIC %,Curr R,Quick R,LTDebt/Eq,Debt/Eq,Gross M %,Oper M %,Profit M %,Perf 3D %,Perf Week %,Perf Month %,Perf Quart %,Perf Half %,Perf Year %,Perf YTD %,Beta,ATR,ATR/Price %,Volatility W %,Volatility M %,SMA20 %,SMA50 %,SMA200 %,50D High %,50D Low %,52W High %,52W Low %,52W Range,All-Time High %,All-Time Low %,RSI,Earnings,IPO Date,Optionable,Shortable,Employees,Change from Open %,Gap %,Recom,"Avg Volume, M",Rel Volume,Volume,Target Price,Prev Close,Open,High,Low,Price,Change %,Single Category,Asset Type,Expense %,Holdings,"AUM, M","Flows 1M, M",Flows% 1M,"Flows 3M, M",Flows% 3M,"Flows YTD, M",Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags,Sharpe 3d,Sortino 3d,Omega 3d,Sharpe 5d,Sortino 5d,Omega 5d,Sharpe 10d,Sortino 10d,Omega 10d,Sharpe 15d,Sortino 15d,Omega 15d,Sharpe 30d,Sortino 30d,Omega 30d,Sharpe 60d,Sortino 60d,Omega 60d,Sharpe 120d,Sortino 120d,Omega 120d,Sharpe 250d,Sortino 250d,Omega 250d
VV,20,Visa Inc,"DJIA, S&P 500",Financial,Credit Services,USA,NYSE,"Financial, Credit Services",675570.0,23,675570.0,31.52,NaN,1.76,15.70,19.23,42.43,31.89,18.64,8.45,0.76,2.60,5/12/2026,23.37,11.37,3.23,NaN,NaN,NaN,NaN,NaN,17.05,35.82,13.59,14.37,43030.0,22040.0,6.79,4.46,1660.0,1660.0,99.75,12.11,-0.03,80.87,-1.20,1.25,2.45,20.68,23.45,59.80,37.94,1.09,1.09,0.63,0.67,78.28,67.24,51.21,-1.241770,2.75,7.64,13.79,8.93,2.50,2.24,0.75,8.33,2.323182,2.36,2.31,3.54,7.64,8.78,-1.80,16.04,-1.80,22.00,293.89 - 365.14,-4.51,3332.84,62.31,Jul 28/a,3/19/2008,Yes,Yes,34100.0,-1.38,-0.43,1.33,8.42,0.88,7400420,404.92,365.14,363.57,364.63,357.00,358.56,-1.80,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-46.792230,-15.436556,0.000000,-5.177200,-6.089985,0.442069,-3.562795,-4.449499,0.597694,2.244057,3.847987,1.400356,-1.107443,-1.476726,0.834344,1.291966,1.843792,1.234719,0.896235,1.329832,1.157487,1.129936,1.629317,1.206749
PPG,38,Procter & Gamble Co,"DJIA, S&P 500",Consumer Defensive,Household & Personal Products,USA,NYSE,"Consumer Defensive, Household & Personal Products",349240.0,43,349240.0,21.94,NaN,7.19,4.03,6.50,28.38,23.24,23.08,5.28,2.84,4.26,7/24/2026,62.63,6.84,1.41,NaN,NaN,NaN,NaN,NaN,7.38,5.77,8.06,3.33,86720.0,16320.0,2.20,3.43,2330.0,2330.0,99.92,0.08,-15.76,72.38,0.09,1.03,2.64,23.95,13.22,31.12,21.21,0.73,0.53,0.44,0.68,50.88,24.13,18.82,2.354788,2.00,-1.65,4.80,2.48,-3.62,4.65,0.37,3.37,2.246966,2.50,2.04,0.63,2.28,1.17,-2.29,8.01,-10.33,8.98,137.62 - 167.25,-16.88,4702.95,53.14,Jul 29/b,3/22/1950,Yes,Yes,109000.0,-1.88,0.90,2.07,9.07,0.98,8917395,162.67,151.50,152.86,154.32,149.31,149.98,-1.00,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,4.948079,17.697306,2.576601,7.274221,21.347154,3.689489,-3.802228,-4.427376,0.537767,-2.469434,-2.977591,0.673347,1.241257,1.783730,1.212618,0.765010,1.125972,1.125179,0.379260,0.571765,1.062696,0.186491,0.270771,1.031860
TTD,65,Toronto Dominion Bank,-,Financial,Banks - Diversified,Canada,NYSE,"Financial, Banks - Diversified",208820.0,72,208820.0,20.12,NaN,1.21,2.55,2.47,NaN,6.39,50.12,NaN,2.56,3.10,7/10/2026,36.30,6.14,1.76,NaN,NaN,NaN,NaN,NaN,-23.00,-59.81,-10.02,-10.74,81980.0,10390.0,4.70,6.21,1690.0,1690.0,99.77,0.23,0.00,51.11,-2.90,0.46,3.32,7.84,0.71,11.84,8.62,0.54,NaN,0.37,2.22,NaN,17.02,12.67,-1.847941,2.55,4.56,18.80,31.83,67.03,31.21,0.71,2.07,1.674757,1.62,1.63,2.26,7.05,26.58,-1.02,16.87,-1.02,71.17,72.21 - 124.87,-1.02,2519.34,64.40,May 28/b,8/30/1996,Yes,Yes,NaN,0.42,-0.66,2.25,2.36,0.65,1524498,113.48,123.90,123.08,124.62,122.26,1